In [8]:
import requests, re, io
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

In [14]:
CCRS_PAGE_URL = "https://data.ca.gov/dataset/ccrs" # da url
HEADERS = {"User-Agent": "CS122-TripSafe-NotebookTest/1.0"} #so before we were getting 404 error due to default header made by request, this allows for the site to know that were not a bot rather a dedicated user

print("Environment setup complete.")

Environment setup complete.


In [28]:
response = requests.get(CCRS_PAGE_URL, headers=HEADERS, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
links = []

for a in soup.find_all("a", href=True):
    href = urljoin(CCRS_PAGE_URL, a["href"])
    text = a.get_text(strip=True)
    if ".csv" in href.lower() and any(k in href.lower() for k in ["crashes_", "parties_"]):
        links.append({"name": text, "url": href})

print(f"Found {len(links)} potential data links.")
for a in links:
    print(a)


Found 39 potential data links.
{'name': 'Download', 'url': 'https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/9f4fc839-122d-4595-a146-43bc4ed16f46/download/crashes_2025.csv'}
{'name': 'Download', 'url': 'https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/a2676918-a825-4b77-8e5c-6eadb38d6b1a/download/parties_2025.csv'}
{'name': 'Download', 'url': 'https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/f775df59-b89b-4f82-bd3d-8807fa3a22a0/download/crashes_2024.csv'}
{'name': 'Download', 'url': 'https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/93892d36-017b-4a2a-bc0b-f1f385060b96/download/parties_2024.csv'}
{'name': 'Download', 'url': 'https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/436642c0-cd04-4a4c-b45e-564b66437476/download/crashes_2023.csv'}
{'name': 'Download', 'url': 'https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/84376be5-548b-44e3-8ebc-73

In [31]:
processed = []
for item in links:
    year_match = re.search(r"20\d{2}", f"{item['name']} {item['url']}")
    if(year_match):
        year = year_match.group(0) 
        processed.append({        
        "Display Name": item['name'] if item['name'] else "CCRS Data",
        "Year": year,
        "Download URL": item['url']
    })
    else :
        "Unknown"

#creates a data frame of the years of csvs
datadf = pd.DataFrame(processed).drop_duplicates(subset=["Download URL"])
datadf.to_csv("ccrs_resource_catalog.csv", index=False)

print("links to all data have been saved")
catalog_df.head()

links to all data have been saved


,Display Name,Year,Download URL
0,Download,2025,https://data.ca.gov/dataset/80c6a49d-c6b3-40ba...
1,Download,2025,https://data.ca.gov/dataset/80c6a49d-c6b3-40ba...
2,Download,2025,https://data.ca.gov/dataset/80c6a49d-c6b3-40ba...
3,Download,2024,https://data.ca.gov/dataset/80c6a49d-c6b3-40ba...
4,Download,2024,https://data.ca.gov/dataset/80c6a49d-c6b3-40ba...


In [35]:
if not datadf.empty:
    url = catalog_df.iloc[0]['Download URL']   
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status() 
        mainData = io.StringIO(response.text)
        data = pd.read_csv(mainData, low_memory=False)
        
        if mainData.empty:
            print("Warning: File exists but contains no data rows (headers only).")
        else:
            print("Success: Data accessed successfully!")
            local_name = f"{}.csv"
            sample_data.to_csv(local_name, index=False)
            print(f"Sample saved locally as: {local_name}")
            
            display(sample_data.head())
            
    except requests.exceptions.HTTPError as e:
        print(f"Server Error: {e}. The site may be temporarily blocking automated access.")
    except Exception as e:
        print(f" Unexpected Error: {e}")

✅ Success: Data accessed successfully!
Sample saved locally as: raw_crash_data_sample.csv


,Collision Id,\tReport Number,\tReport Version,\tIs Preliminary,\tNCIC Code,Crash Date Time,\tCrash Time Description,\tBeat,City Id,\tCity Code,\tCity Name,County Code,\tCity Is Active,\tCity Is Incorporated,\tCollision Type Code,Collision Type Description,\tCollision Type Other Desc,\tDay Of Week,\tDispatchNotified,\tHasPhotographs,\tHitRun,\tIsAttachmentsMailed,\tIsDeleted,\tIsHighwayRelated,\tIsTowAway,\tJudicialDistrict,\tMotorVehicleInvolvedWithCode,\tMotorVehicleInvolvedWithDesc,MotorVehicleInvolvedWithOtherDesc,\tNumberInjured,\tNumberKilled,Weather 1,Weather 2,Road Condition 1,Road Condition 2,Special Condition,LightingCode,LightingDescription,Latitude,Longitude,MilepostDirection,MilepostDistance,MilepostMarker,MilepostUnitOfMeasure,\tPedestrianActionCode,PedestrianActionDesc,PreparedDate,\tPrimary Collision Factor Code,Primary Collision Factor Violation,PrimaryCollisionFactorIsCited,\tPrimaryCollisionPartyNumber,\tPrimaryRoad,\tReportingDistrict,\tReportingDistrictCode,ReviewedDate,\tRoadwaySurfaceCode,SecondaryDirection,\tSecondaryDistance,\tSecondaryRoad,\tSecondaryUnitOfMeasure,\tSketchDesc,\tTrafficControlDeviceCode,CreatedDate,\tModifiedDate,IsCountyRoad,IsFreeway,CHP555Version,\tIsAdditonalObjectStruck,NotificationDate,\tNotificationTimeDescription,\tHasDigitalMediaFiles,\tEvidenceNumber,\tIsLocationReferToNarrative,\tIsAOIOneSameAsLocation
0,4550266,9680-2025-00147,1,False,9680,1/14/2025 7:50:00 AM,750,027,1219,3700,Unincorporated,37,True,False,C,REAR END,NaN,Tuesday,NotApplicable,False,NaN,NaN,False,True,True,SAN DIEGO SUPERIOR COURT EAST COUNTY REGIONAL ...,C,OTHER MOTOR VEHICLE,NaN,0,0,CLEAR,NaN,NO UNUSUAL CONDITIONS,NaN,,A,DAYLIGHT,32.742237,-117.050468,NaN,NaN,NaN,NaN,A,NO PEDESTRIANS INVOLVED,1/14/2025 7:50:00 AM,A,VC 22350,False,1,SR-94 W/B,NaN,NaN,1/27/2025 9:04:35 AM,A,E,120.0,COLLEGE AVE,F,NaN,D,1/27/2025 9:04:38 AM,1/27/2025 9:04:38 AM,False,True,4,NaN,1/14/2025 7:55:00 AM,755.0,True,EMBEDDED,NaN,True
1,4550265,9685-2025-00096,1,False,9685,1/18/2025 3:20:00 PM,1520,039,994,3300,Unincorporated,33,True,False,F,OVERTURNED,NaN,Saturday,Yes,False,NaN,NaN,False,True,True,RIVERSIDE SUPERIOR COURT SOUTHWEST JUSTICE CEN...,A,NON-COLLISION,NaN,1,0,CLEAR,NaN,NO UNUSUAL CONDITIONS,NaN,,A,DAYLIGHT,33.487271,-117.050273,NaN,NaN,NaN,NaN,A,NO PEDESTRIANS INVOLVED,1/18/2025 3:20:00 PM,A,VC 22107,False,1,SR-79 N/B,NaN,NaN,1/27/2025 9:04:35 AM,A,S,1710.0,ANZA ROAD,F,NaN,D,1/27/2025 9:04:38 AM,1/27/2025 9:04:38 AM,False,False,4,NaN,1/18/2025 3:25:00 PM,1525.0,False,NaN,NaN,True
2,4550264,250110005,1,False,4116,1/10/2025 8:28:00 AM,828,A,1310,4116,San Mateo,41,True,True,C,REAR END,NaN,Friday,NotApplicable,True,NaN,NaN,False,False,False,NORTHERN TRAFFIC,C,OTHER MOTOR VEHICLE,NaN,1,0,CLEAR,NaN,NO UNUSUAL CONDITIONS,NaN,,A,DAYLIGHT,37.567010,-122.330000,NaN,NaN,NaN,NaN,A,NO PEDESTRIANS INVOLVED,NaN,A,22350 VC,False,1,N EL CAMINO REAL,13A,NaN,NaN,A,NaN,NaN,TILTON AVENUE,NaN,NaN,A,1/27/2025 9:04:06 AM,1/27/2025 9:04:06 AM,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4550263,9580-2025-00214,1,False,9580,1/23/2025 8:45:00 AM,845,171,535,1942,Los Angeles,19,True,True,B,SIDE SWIPE,NaN,Thursday,NotApplicable,False,M,NaN,False,True,False,LOS ANGELES SUPERIOR COURT VAN NUYS COURTHOUSE...,C,OTHER MOTOR VEHICLE,NaN,0,0,CLEAR,NaN,NO UNUSUAL CONDITIONS,NaN,,A,DAYLIGHT,34.199578,-118.403607,NaN,NaN,NaN,NaN,A,NO PEDESTRIANS INVOLVED,1/23/2025 8:45:00 AM,A,VC 22107,False,1,SR-170 S/B FROM SHERMAN WAY,NaN,NaN,1/27/2025 9:03:34 AM,A,S,500.0,SHERMAN WAY,F,NaN,D,1/27/2025 9:03:36 AM,1/27/2025 9:03:36 AM,False,True,4,NaN,1/23/2025 8:47:00 AM,847.0,False,NaN,NaN,True
4,4550262,9765-2025-00066,1,False,9765,1/15/2025 12:30:00 PM,1230,041,1701,5606,Santa Paula,56,True,True,E,HIT OBJECT,NaN,Wednesday,NotApplicable,False,NaN,NaN,False,True,False,VENTURA SUPERIOR COURT VENTURA DIVISION,J,OTHER OBJECT,METAL OBJECT,0,0,WIND,NaN,NO UNUSUAL CONDITIONS,NaN,,A,DAYLIGHT,34.333348,-119.087588,NaN,NaN,NaN,NaN,A,NO PEDESTRIANS INVOLVED,1/15/2025 12:30:00 PM,A,VC 223